### Setup

In [ ]:
import os
import sys
sys.path.insert(0, '.')

try:
    import google.colab
    from google.colab import drive

    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

    colab_src_dir = "/content/drive/MyDrive/Luca/Research/AI-in-Science-Lab/greedy-stacked-autoencoders/src"
    os.chdir(colab_src_dir)
    if colab_src_dir not in sys.path:
        sys.path.insert(0, colab_src_dir)

In [ ]:
import copy
import math
import numpy as np
import torch
import torch.nn as nn
import wandb
from torch.utils.data import DataLoader
from torchvision.utils import make_grid
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from models import Autoencoder, K_Sparse_AE, WTA_FC_AE, WTA_CONV_AE
from datasets import get_data_loader, get_patch_shape

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {device}")

In [ ]:
wandb.init(
    project="filter-deduplicate",
    config={
        "k_population": 0.05,
        "k_spatial": 0.2,
        "k_lifetime": 0.2,
        "epochs": 30,
        "batch_size": 128,
        "bottleneck_dim": 250,
        "lr": 0.001,
        "dataset": "cifar10_color",
        "autoencoder_type": "normal",
    },
)

### Dataset

In [ ]:
os.makedirs('../data', exist_ok=True)

In [ ]:
config = wandb.config

In [ ]:
def get_dataset_size(dataset: str) -> int:
    match dataset:
        case "mnist_patches" | "cifar10_patches":
            return 8 * 8
        case "cifar10_patches_color":
            return 8 * 8 * 3
        case "mnist":
            return 28 * 28
        case "cifar10":
            return 32 * 32
        case "cifar10_color":
            return 32 * 32 * 3
        case _:
            raise ValueError(f"Unknown dataset: {dataset!r}")

In [ ]:
def get_num_train_samples(dataset: str) -> int:
    match dataset:
        case "mnist":
            return 60_000
        case "mnist_patches":
            return 10_000
        case "cifar10":
            return 50_000
        case "cifar10_patches":
            return 10_000
        case "cifar10_patches_color":
            return 1_000_000
        case "cifar10_color":
            return 50_000
        case _:
            raise ValueError(f"Unknown dataset: {dataset!r}")

### Autoencoders

In [ ]:
def get_autoencoder(autoencoder_type: str, config, device):
    input_dim = get_dataset_size(config.dataset)
    n_train   = get_num_train_samples(config.dataset)
    match autoencoder_type:
        case "normal":
            return Autoencoder(
                dim=(input_dim, config.bottleneck_dim),
            ).to(device)

        case "sparse":
            return K_Sparse_AE(
                dim=(input_dim, config.bottleneck_dim),
                k_population=config.k_population,
                total_epochs=config.epochs,
                dataset_size=n_train,
                a=5,
            ).to(device)

        case "wta_fc":
            return WTA_FC_AE(
                dim=(input_dim, config.bottleneck_dim),
                k_lifetime=config.k_lifetime,
            ).to(device)

        case "wta_conv":
            try:
                c, h, w = get_patch_shape(config.dataset)
            except ValueError:
                s = int(math.sqrt(input_dim))
                c, h, w = 1, s, s

            return WTA_CONV_AE(
                dim=(c, h, w, config.bottleneck_dim),
                k_spatial=config.k_spatial,
                k_population=config.k_lifetime,
                total_epochs=config.epochs,
                dataset_size=n_train,
                a=5,
            ).to(device)

        case _:
            raise ValueError(f"Unknown autoencoder_type={autoencoder_type!r}.")

### Criterion

In [ ]:
class Redundancy(nn.Module):
    
    @staticmethod
    def get_correlation_measure(correlation: str):
        def r_squared(v1, v2):
            v1c, v2c = v1 - v1.mean(), v2 - v2.mean()
            r = (v1c * v2c).sum() / (v1c.norm() * v2c.norm()).clamp_min(1e-12)
            return r * r

        def cosine_similarity(v1, v2):
            return (v1 * v2).sum() / (v1.norm() * v2.norm()).clamp_min(1e-12)

        match correlation:
            case "r_squared":
                return r_squared
            case "cosine_similarity":
                return cosine_similarity
            case _:
                raise ValueError(f"Unknown correlation={correlation!r}.")
            
            
    @staticmethod
    def get_falloff_measure(falloff: str):
        def exponential_decay(dw, dh):
            d = torch.hypot(dw, dh)
            return torch.exp(-d)

        def linear_decay(dw, dh):
            d = torch.hypot(dw, dh)
            return torch.clamp(1.0 - d, min=0.0)
        
        def no_decay(dw, dh):
            return torch.ones_like(dw)

        match falloff:
            case "exponential_decay":
                return exponential_decay
            case "linear_decay":
                return linear_decay
            case "no_decay":
                return no_decay
            case _:
                raise ValueError(f"Unknown falloff={falloff!r}.")
    
    
    @staticmethod
    def get_evaluation_type(evaluation: str, correlation: str, falloff: str, autoencoder_type: str):
        
        def full_evaluation(correlation, falloff, hidden_output):
            correlation_measure = Redundancy.get_correlation_measure(correlation)
            falloff_measure     = Redundancy.get_falloff_measure(falloff)

            match autoencoder_type:
                case "conv":
                    if hidden_output.ndim != 4:
                        raise ValueError(
                            f"'autoencoder_type=conv' expects hidden_output ndim=4 (B,C,H,W); got "
                            f"{hidden_output.ndim=} shape={tuple(hidden_output.shape)}"
                        )
                        
                    B, C, H, W = hidden_output.shape
                    
                    redundancy_sum = hidden_output.new_zeros(())
                    channel_pairs  = (C**2) / 2
                    location_pairs = ((H * W) ** 2) / 2
                    for c1 in range(C):
                        for c2 in range(c1 + 1, C):
                            for p1 in range(H * W):
                                h1, w1 = divmod(p1, W)
                                for p2 in range(p1 + 1, H * W):
                                    h2, w2 = divmod(p2, W)

                                    v1 = hidden_output[:, c1, h1, w1]
                                    v2 = hidden_output[:, c2, h2, w2]
                                    dw = hidden_output.new_tensor(w2 - w1, dtype=hidden_output.dtype)
                                    dh = hidden_output.new_tensor(h2 - h1, dtype=hidden_output.dtype)
                                    correlation = correlation_measure(v1, v2)
                                    falloff     = falloff_measure(dw, dh)
                                    redundancy_sum = redundancy_sum + falloff * correlation

                    redundancy = redundancy_sum / (channel_pairs * location_pairs)
                    return redundancy

                case _:
                    raise ValueError(f"Unknown autoencoder_type={autoencoder_type!r}.")

        match evaluation:
            case "full_evaluation":
                return full_evaluation
            case _:
                raise ValueError(f"Unknown evaluation={evaluation!r}.")
            
    def __init__(
        self,
        evaluation: str,
        correlation: str,
        falloff: str,
        autoencoder_type: str,
        lambda_weight: float = 1.0,
    ) -> None:
        super().__init__()
        self.mse           = nn.MSELoss()
        self.redundancy    = Redundancy.get_evaluation_type(evaluation, correlation, falloff, autoencoder_type)
        self.lambda_weight = lambda_weight
        

    def forward(
        self,
        input: torch.Tensor,
        target: torch.Tensor,
        hidden_output: torch.Tensor | None = None,
    ) -> torch.Tensor:
        mse        = self.mse(input, target)
        redundancy = self.redundancy(self.correlation, self.falloff, hidden_output)
        return mse + self.lambda_weight * redundancy

### Training

In [ ]:
def train(config):
    data_loader = get_data_loader(config.dataset, train=True, batch_size=config.batch_size)
    test_loader = get_data_loader(config.dataset, train=False, batch_size=config.batch_size)
    autoencoder = get_autoencoder(config.autoencoder_type, config, device)
    optimizer   = torch.optim.Adam(autoencoder.parameters(), lr=config.lr)

    redundancy_autoencoder_type = "conv" if getattr(autoencoder, "is_convolutional", False) else "fc"
    criterion = Redundancy(
        evaluation=config.evaluation,
        correlation=config.correlation,
        falloff=config.falloff,
        autoencoder_type=redundancy_autoencoder_type,
        lambda_weight=config.lambda_weight
    )
    

    def compute_test_loss(cur_epoch: int) -> float:
        autoencoder.eval()
        test_loss_sum = 0.0
        test_n = 0
        with torch.no_grad():
            for batch_inputs, _ in test_loader:
                batch_inputs = batch_inputs.to(device)
                if autoencoder.uses_k_population:
                    model_out = autoencoder(
                        batch_inputs,
                        epoch=cur_epoch,
                        inputs_processed_in_epoch=0,
                    )
                else:
                    model_out = autoencoder(batch_inputs)

                if isinstance(model_out, tuple):
                    x_recon, hidden_output = model_out
                else:
                    x_recon, hidden_output = model_out, None

                test_loss_sum += (
                    criterion(x_recon, batch_inputs, hidden_output).item() * batch_inputs.shape[0]
                )
                test_n += batch_inputs.shape[0]
        autoencoder.train()
        return test_loss_sum / max(test_n, 1)

    losses            = []
    model_checkpoints = []
    autoencoder.train()

    epochbar  = tqdm(range(config.epochs), desc="Epochs")
    n_batches = len(data_loader)
    log_every = max(1, n_batches // 4)

    for epoch in epochbar:
        epoch_loss                = 0.0
        inputs_processed_in_epoch = 0

        for step, (batch_inputs, _) in enumerate(data_loader):
            batch_size_cur = batch_inputs.shape[0]
            batch_inputs   = batch_inputs.to(device)
            optimizer.zero_grad()

            if autoencoder.uses_k_population:
                model_out = autoencoder(
                    batch_inputs,
                    epoch=epoch,
                    inputs_processed_in_epoch=inputs_processed_in_epoch,
                )
            else:
                model_out = autoencoder(
                    batch_inputs
                )


            x_recon, hidden_output = model_out

            inputs_processed_in_epoch += batch_size_cur

            loss = criterion(x_recon, batch_inputs, hidden_output)
            loss.backward()
            optimizer.step()

            batch_loss = loss.item()
            epoch_loss += batch_loss

            if (step + 1) % log_every == 0:
                avg = epoch_loss / (step + 1)
                losses.append(avg)
                wandb.log({
                    "test/loss": compute_test_loss(epoch),
                    "epoch": epoch,
                    "train/step_in_epoch": step,
                })

            epochbar.set_postfix({config.autoencoder_type: f"{batch_loss:.4f}"})

        epoch_mean     = epoch_loss / n_batches
        test_loss_mean = compute_test_loss(epoch)

        wandb.log({
            "train/loss": epoch_mean,
            "test/loss": test_loss_mean,
            "epoch": epoch,
        })

        model_checkpoints.append({config.autoencoder_type: copy.deepcopy(autoencoder)})

    return autoencoder, model_checkpoints, losses, data_loader, test_loader

### Visualizations


In [ ]:
def log_figure_wandb(name: str, fig=None) -> None:
    
    if fig is None:
        fig = plt.gcf()
        
    fig.canvas.draw()
    
    rgba = np.asarray(fig.canvas.buffer_rgba())
    rgb  = rgba[..., :3]
    wandb.log({name: wandb.Image(rgb)})

In [ ]:
def _chw_from_config(config, input_dim: int) -> tuple[int, int, int]:
    try:
        return get_patch_shape(config.dataset)
    except ValueError:
        s = int(math.sqrt(input_dim))
        return 1, s, s

In [ ]:
def viz_sample_patches(config, data_loader: DataLoader) -> None:
    patches_batch, _ = next(iter(data_loader))
    n_show           = min(25, patches_batch.size(0))
    input_dim        = get_dataset_size(config.dataset)

    C, H, W      = _chw_from_config(config, input_dim)
    patches_vis  = patches_batch[:n_show].reshape(-1, C, H, W)
    grid_patches = make_grid(patches_vis, nrow=5, normalize=True, scale_each=True, padding=2)

    fig = plt.figure(figsize=(8, 8))
    plt.imshow(grid_patches.permute(1, 2, 0).clamp(0, 1))
    plt.axis("off")
    plt.title(f"Sample {config.dataset} patches (n={n_show})")
    plt.tight_layout()
    log_figure_wandb("visualizations/dataset/sample_patches", fig)
    plt.show()

In [ ]:
def viz_encoder_weights(
    config,
    model_checkpoints: list,
) -> None:
    def mask_linear_encoder_rows(W: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        m_row = mask.view(-1)
        return W * m_row.unsqueeze(1)

    def mask_conv_encoder_out(W: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        m_row = mask.view(-1)
        return W * m_row.view(-1, 1, 1, 1)

    visualize_epoch     = config.epochs - 1
    input_dim           = get_dataset_size(config.dataset)
    max_encoder_filters = 300

    C, H, W_img = _chw_from_config(config, input_dim)
    cp          = model_checkpoints[visualize_epoch]
    m           = cp[config.autoencoder_type]

    mask_t = m.last_filter_mask[0].cpu()
    max_filters = min(max_encoder_filters, m.detached_encoder_weights.shape[0])
    if m.is_convolutional:
        W_enc = mask_conv_encoder_out(m.detached_encoder_weights.cpu(), mask_t)
        kernels = W_enc[:max_filters]
    else:
        W_enc = mask_linear_encoder_rows(m.detached_encoder_weights.cpu(), mask_t)
        kernels = W_enc[:max_filters].reshape(-1, C, H, W_img)
    grid = make_grid(kernels, nrow=10, normalize=True, scale_each=True, padding=1)

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(grid.permute(1, 2, 0).clamp(0, 1))
    ax.axis("off")
    ax.set_title(f"{config.autoencoder_type} - masked encoder weights")
    fig.suptitle(f"Epoch {visualize_epoch}, probe batch n=1")
    plt.tight_layout()
    log_figure_wandb(f"visualizations/encoder_weights/{config.autoencoder_type}", fig)
    plt.show()

In [ ]:
def viz_decoder_weights(
    config, model_checkpoints: list
) -> None:
    visualize_epoch     = config.epochs - 1
    input_dim           = get_dataset_size(config.dataset)
    max_decoder_filters = 300

    C, H, W_img = _chw_from_config(config, input_dim)
    cp          = model_checkpoints[visualize_epoch]
    m           = cp[config.autoencoder_type]

    W_dec       = m.detached_decoder_weights.cpu()
    max_filters = min(max_decoder_filters, W_dec.shape[0])
    if m.is_convolutional:
        kernels = W_dec[:max_filters]
    else:
        kernels = W_dec.T[:max_filters].reshape(-1, C, H, W_img)
    grid = make_grid(kernels, nrow=10, normalize=True, scale_each=True, padding=1)

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(grid.permute(1, 2, 0).clamp(0, 1))
    ax.axis("off")
    ax.set_title(f"{config.autoencoder_type} - decoder weights")
    plt.suptitle(f"Epoch {visualize_epoch}")
    plt.tight_layout()
    log_figure_wandb(f"visualizations/decoder_weights/{config.autoencoder_type}", fig)
    plt.show()

In [ ]:
def viz_reconstructions(
    config,
    device: torch.device,
    data_loader: DataLoader,
    model_checkpoints: list,
) -> None:
    visualize_epoch = config.epochs - 1
    input_dim       = get_dataset_size(config.dataset)

    C, H, W = _chw_from_config(config, input_dim)
    cp      = model_checkpoints[visualize_epoch]
    m       = cp[config.autoencoder_type]
    m.eval()

    batch_inputs, _ = next(iter(data_loader))
    idx             = torch.randperm(batch_inputs.shape[0])[:25]
    inputs_25       = batch_inputs[idx].to(device)
    with torch.no_grad():
        if m.uses_k_population:
            recon = m(
                inputs_25,
                epoch=visualize_epoch,
                inputs_processed_in_epoch=0,
            ).cpu()
        else:
            recon = m(inputs_25).cpu()
    orig_vis  = inputs_25.cpu().reshape(-1, C, H, W)
    rec_vis   = recon.reshape(-1, C, H, W)
    grid_orig = make_grid(orig_vis, nrow=5, normalize=True, scale_each=True, padding=2)
    grid_rec  = make_grid(rec_vis, nrow=5, normalize=True, scale_each=True, padding=2)

    fig, axes = plt.subplots(2, 1, figsize=(8, 8))
    axes[0].imshow(grid_orig.permute(1, 2, 0).clamp(0, 1))
    axes[0].axis("off")
    axes[0].set_title("Original patches (25)")
    axes[1].imshow(grid_rec.permute(1, 2, 0).clamp(0, 1))
    axes[1].axis("off")
    axes[1].set_title(f"{config.autoencoder_type} reconstructions")
    plt.tight_layout()
    log_figure_wandb(f"visualizations/reconstructions/{config.autoencoder_type}", fig)
    plt.show()

### Training

In [ ]:
config = wandb.config
(
    autoencoder,
    model_checkpoints,
    losses,
    data_loader,
    test_loader,
    input_dim,
) = train(config)

In [ ]:
viz_sample_patches(config, data_loader)
viz_encoder_weights(config, model_checkpoints)
viz_decoder_weights(config, model_checkpoints)
viz_reconstructions(config, device, data_loader, model_checkpoints)